In [ ]:
# autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
from depth.depth_anything_model import DepthAnythingModel
from depth.midas_model import MidasModel
from image import Image
from image_processing import unletterbox
import numpy as np
import cv2
import matplotlib.pyplot as plt
from time import perf_counter
from tqdm import tqdm

In [ ]:
model = DepthAnythingModel("depth_anything_vits14", "mps")

In [ ]:
def frame_iterator(filepath: str):
    vid = cv2.VideoCapture(filepath)
    while True:
        ret, raw_frame = vid.read()
        if not ret:
            break
        frame = cv2.cvtColor(raw_frame, cv2.COLOR_BGR2RGB)
        yield frame
    vid.release()

def frame_batch_iterator(filepath: str, batch_size: int = 8):
    vid = cv2.VideoCapture(filepath)
    while True:
        frames = []
        for _ in range(batch_size):
            ret, raw_frame = vid.read()
            if not ret:
                break
            frame = cv2.cvtColor(raw_frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        if not frames:
            break
        yield np.asarray(frames)
    vid.release()

In [ ]:
video_path = "path_to_video.mp4"

In [ ]:
from time import perf_counter
for batch in frame_batch_iterator(video_path, 2):
    start = perf_counter()
    depth = model.predict_batch_depth(batch)
    print(f'Elapsed: {perf_counter() - start:.2f}s')
    break

In [ ]:
def generate_depth_video(video_path: str, output_path: str):
    vid = cv2.VideoCapture(video_path)
    width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = vid.get(cv2.CAP_PROP_FPS)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    num_frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))

    for i in tqdm(range(num_frames)):
        ret, frame = vid.read()
        if not ret:
            break
        depth = model.predict_depth(frame)
        # Ensure the depth map is 3 channels and matches the video frame size
        depth = cv2.cvtColor(depth, cv2.COLOR_GRAY2BGR)
        depth = cv2.resize(depth, (width, height))
        out.write(depth)
        if i > 100:  # For testing, remove or adjust for the full video
            break

    vid.release()
    out.release()


In [ ]:
generate_depth_video(video_path, "depth_video_2.mp4")

In [ ]:
for frame in frame_iterator(video_path):
    start = perf_counter()
    depth = model.predict_depth(frame)
    print(f'Elapsed: {perf_counter() - start:.2f}s')
    plt.imshow(depth)
    plt.show()